In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import plotly.express as px
import logging
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

mpl.rcParams['figure.dpi'] = 150
plt.style.use('seaborn-v0_8-whitegrid')
logging.disable(logging.CRITICAL)

In [2]:
df = pd.read_csv('../data/eu_regional_data.csv', index_col = 'geo')
df.head()

,region_name,Country,EU Region,GDP per Capita,GDP per Capita (PPS),Unemployment %,Employment in Hi-tech Sectors %,Life Expectancy,Doctors per 100K,Heart Disease Deaths per 100K,Cancer Deaths per 100K,Fatal Road Accidents per Million,Tertiary Educational Attainment %,Population Density,People at Risk of Poverty %,Regular Internet Users %,Land Covered by Buildings and Roads %,Land Covered by Crops %
geo,,,,,,,,,,,,,,,,,,
BE10,Région de Bruxelles-Capitale/ Brussels Hoofdst...,BE,Western Europe,82300.0,190.0,10.6,7.0,82.1,518.68,49.84,211.03,5.0,53.8,7770.2,38.9,91.10,37.0,2.5
BE21,Prov. Antwerpen,BE,Western Europe,60400.0,140.0,3.6,6.1,83.3,294.48,44.22,204.50,36.0,46.1,686.3,15.2,90.81,17.6,23.5
BE22,Prov. Limburg (BE),BE,Western Europe,41300.0,96.0,3.2,4.1,83.2,281.87,42.19,202.63,36.0,42.2,377.0,11.7,91.27,12.6,26.5
BE23,Prov. Oost-Vlaanderen,BE,Western Europe,48200.0,111.0,2.9,5.9,82.7,315.01,44.26,216.72,44.0,46.8,528.3,11.5,91.56,17.3,26.4
BE24,Prov. Vlaams-Brabant,BE,Western Europe,55700.0,129.0,4.2,7.2,83.8,289.13,43.10,200.59,28.0,50.0,564.3,11.0,94.14,14.6,30.3


In [3]:
df.isna().sum()

region_name                               0
Country                                   0
EU Region                                 0
GDP per Capita                            0
GDP per Capita (PPS)                      0
Unemployment %                            5
Employment in Hi-tech Sectors %          19
Life Expectancy                           0
Doctors per 100K                         60
Heart Disease Deaths per 100K             0
Cancer Deaths per 100K                    0
Fatal Road Accidents per Million         13
Tertiary Educational Attainment %         1
Population Density                        2
People at Risk of Poverty %              12
Regular Internet Users %                 54
Land Covered by Buildings and Roads %    10
Land Covered by Crops %                  10
dtype: int64

In [4]:
df.columns

Index(['region_name', 'Country', 'EU Region', 'GDP per Capita',
       'GDP per Capita (PPS)', 'Unemployment %',
       'Employment in Hi-tech Sectors %', 'Life Expectancy',
       'Doctors per 100K', 'Heart Disease Deaths per 100K',
       'Cancer Deaths per 100K', 'Fatal Road Accidents per Million',
       'Tertiary Educational Attainment %', 'Population Density',
       'People at Risk of Poverty %', 'Regular Internet Users %',
       'Land Covered by Buildings and Roads %', 'Land Covered by Crops %'],
      dtype='object')

In [5]:
cols_num = ['GDP per Capita', 'GDP per Capita (PPS)',
       'Unemployment %', 'Employment in Hi-tech Sectors %', 'Life Expectancy',
       'Doctors per 100K', 'Heart Disease Deaths per 100K',
       'Cancer Deaths per 100K', 'Fatal Road Accidents per Million',
       'Tertiary Educational Attainment %', 'Population Density',
       'People at Risk of Poverty %', 'Regular Internet Users %',
       'Land Covered by Buildings and Roads %', 'Land Covered by Crops %']

imputer = KNNImputer()
df[cols_num] = imputer.fit_transform(df[cols_num])


In [6]:
# scaling variables
scaler = StandardScaler()
X = scaler.fit_transform(df[cols_num])
#X = df[cols_num]

In [7]:
# Principal Component Analysis
pca = PCA(n_components = 2, random_state=0)
components = pca.fit_transform(X)
total_var = pca.explained_variance_ratio_.sum() * 100

df_pca = df.join(pd.DataFrame(components, index = df.index,
                           columns = ['pc_1', 'pc_2']))
df_pca.reset_index(inplace = True)

fig = px.scatter(df_pca, x = 'pc_1', y = 'pc_2', color=df_pca['EU Region'],
                 width = 800, height = 600, size = 'GDP per Capita',
                 hover_data = ['region_name', 'GDP per Capita'])

fig.update_layout(  margin={"r":1,"t":15,"l":1,"b":1},
                    plot_bgcolor = '#FFFFFF',
                    legend = dict(orientation = 'h', yanchor = 'top'),
                    yaxis_title='', xaxis_title='',
                    title = "")

hovertemplate = '%{customdata[0]}' 
fig.update_traces(hovertemplate=hovertemplate)
fig.show()



In [11]:
# t-SNE
tsne = TSNE(n_components = 2, perplexity = 30, init='random', random_state=0)
components = tsne.fit_transform(X)

df_tsne = df.join(pd.DataFrame(components, index = df.index,
                           columns = ['pc_1', 'pc_2']))

df_tsne.reset_index(inplace = True)

fig = px.scatter(df_tsne, x = 'pc_1', y = 'pc_2', color=df_tsne['EU Region'],
                 width = 800, height = 600, size = 'GDP per Capita',
                 hover_data = ['region_name', 'GDP per Capita'])

fig.update_layout(  margin={"r":1,"t":15,"l":1,"b":1},
                    plot_bgcolor = '#FFFFFF',
                    legend = dict(orientation = 'h', yanchor = 'top'),
                    yaxis_title='', xaxis_title='',
                    title = "")

hovertemplate = '%{customdata[0]}' 
fig.update_traces(hovertemplate=hovertemplate)
fig.show()



In [12]:
# UMAP
umap_model = umap.UMAP(n_components = 2, n_neighbors = 30,
                       min_dist=0.1, random_state = 0)
components = umap_model.fit_transform(X)

df_umap = df.join(pd.DataFrame(components, index = df.index,
                           columns = ['pc_1', 'pc_2']))

df_umap.reset_index(inplace = True)

fig = px.scatter(df_umap, x = 'pc_1', y = 'pc_2', color=df_umap['EU Region'],
                 width = 800, height = 600, size = 'GDP per Capita',
                 custom_data = ['region_name', 'GDP per Capita'])

fig.update_layout(  margin={"r":1,"t":15,"l":1,"b":1},
                    plot_bgcolor = '#FFFFFF',
                    legend = dict(orientation = 'h', yanchor = 'top'),
                    yaxis_title='', xaxis_title='',
                    title = "")

hovertemplate = '%{customdata[0]}' 
fig.update_traces(hovertemplate=hovertemplate)
fig.show()



c:\Users\giann\miniconda3\envs\pycaret\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [13]:
#saving datasets 
df_pca.to_csv('../data/eu_regional_data_pca.csv',
          float_format = '%.2f', encoding = 'utf-8')

df_tsne.to_csv('../data/eu_regional_data_tsne.csv',
          float_format = '%.2f', encoding = 'utf-8')

df_umap.to_csv('../data/eu_regional_data_umap.csv',
          float_format = '%.2f', encoding = 'utf-8')